In [ ]:
# Create DNN: split dataset ---> modeling ---> evaluate model
# Imports
import os
import sys
import numpy as np
from huggingface_hub import login
from tqdm import tqdm
# Add the utils folder to Python path
notebook_dir = os.path.dirname(os.path.abspath("__file__"))
pricer_path = os.path.join(notebook_dir, "../utils")
sys.path.insert(0, pricer_path)
# Class Item to validate the returned data and has lots of helper functions
from pricer.items import Item
from pricer.evaluator import evaluate
from sklearn.feature_extraction.text import CountVectorizer
# Torch dependencies
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import train_test_split

In [2]:
# Login to Hugging Face
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
# Download the batches dataset from Hugging Face 
username = "omarshaarawy11"
dataset = f"{username}/best_deal_03_batches"
train, val, test = Item.from_hub(dataset)
items = train + val + test
print(f"Loaded {len(items):,} items")

Loaded 21,091 items


In [4]:
# Invistigate the dataset
columns = ['title', 
'category', 
'price',
'full',
'weight',
'summary',
'prompt',
'id'
]
for i in columns:
    # Combine item with its attrs
    attr = getattr(items[0], i)
    print(attr)

Lurrose 100Pcs Full Cover Fake Toenails Artificial Transparent Nail Tips Nail Art for DIY
all beauty
6.99
None
0.0
Title: Lurrose 100pcs Full Cover Fake Toenails  
Category: Nail Art & Accessories  
Brand: Lurrose  
Description: High-quality, durable artificial toenails perfect for DIY nail art and manicures.  
Details: Features 100 lightweight, eco-friendly ABS nails that are easy to trim, shape, and fit comfortably to natural toenails, suitable for various styles and perfect as a gift.
None
None


In [5]:
class ResidualBlock(nn.Module):
    def __init__(self, hidden_size, dropout_prob):
        super(ResidualBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout_prob),
            nn.Linear(hidden_size, hidden_size),
            nn.LayerNorm(hidden_size),
        )
        self.relu = nn.ReLU()

    def forward(self, x):
        residual = x
        out = self.block(x)
        out += residual  # Skip connection
        return self.relu(out)


class DeepNeuralNetwork(nn.Module):
    def __init__(self, input_size, num_layers=10, hidden_size=4096, dropout_prob=0.2):
        super(DeepNeuralNetwork, self).__init__()

        # First layer
        self.input_layer = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout_prob),
        )

        # Residual blocks
        self.residual_blocks = nn.ModuleList()
        for i in range(num_layers - 2):
            self.residual_blocks.append(ResidualBlock(hidden_size, dropout_prob))

        # Output layer
        self.output_layer = nn.Linear(hidden_size, 1)

    def forward(self, x):
        x = self.input_layer(x)

        for block in self.residual_blocks:
            x = block(x)

        return self.output_layer(x)


class DeepNeuralNetworkRunner:
    def __init__(self, train, val):
        self.train_data = train
        self.val_data = val
        self.vectorizer = None
        self.model = None
        self.device = None
        self.loss_function = None
        self.optimizer = None
        self.scheduler = None
        self.train_dataset = None
        self.train_loader = None
        self.y_mean = None
        self.y_std = None

        np.random.seed(42)
        torch.manual_seed(42)
        torch.cuda.manual_seed(42)

    def setup(self):
        # Feature selection ---> solit dataset ---> encoding
        self.vectorizer = CountVectorizer(max_features=5000, stop_words="english", binary=True)
        train_documents = [item.summary for item in self.train_data]
        X_train_np = self.vectorizer.fit_transform(train_documents)
        self.X_train = torch.FloatTensor(X_train_np.toarray())
        y_train_np = np.array([float(item.price) for item in self.train_data])
        self.y_train = torch.FloatTensor(y_train_np).unsqueeze(1)

        val_documents = [item.summary for item in self.val_data]
        X_val_np = self.vectorizer.transform(val_documents)
        self.X_val = torch.FloatTensor(X_val_np.toarray())
        y_val_np = np.array([float(item.price) for item in self.val_data])
        self.y_val = torch.FloatTensor(y_val_np).unsqueeze(1)

        y_train_log = torch.log(self.y_train + 1)
        y_val_log = torch.log(self.y_val + 1)
        self.y_mean = y_train_log.mean()
        self.y_std = y_train_log.std()
        self.y_train_norm = (y_train_log - self.y_mean) / self.y_std
        self.y_val_norm = (y_val_log - self.y_mean) / self.y_std

        self.model = DeepNeuralNetwork(self.X_train.shape[1])
        total_params = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        print(f"Deep Neural Network created with {total_params:,} parameters")

        if torch.cuda.is_available():
            self.device = torch.device("cuda")
        elif torch.backends.mps.is_available():
            self.device = torch.device("mps")
        else:
            self.device = torch.device("cpu")

        print(f"Using {self.device}")

        self.model.to(self.device)
        self.loss_function = nn.L1Loss()
        self.optimizer = optim.AdamW(self.model.parameters(), lr=0.001, weight_decay=0.01)
        self.scheduler = CosineAnnealingLR(self.optimizer, T_max=10, eta_min=0)

        self.train_dataset = TensorDataset(self.X_train, self.y_train_norm)
        self.train_loader = DataLoader(self.train_dataset, batch_size=64, shuffle=True)

    def train(self, epochs=5):
        # Modeling
        for epoch in range(1, epochs + 1):
            self.model.train()
            train_losses = []

            for batch_X, batch_y in tqdm(self.train_loader):
                batch_X = batch_X.to(self.device)
                batch_y = batch_y.to(self.device)

                self.optimizer.zero_grad()
                outputs = self.model(batch_X)
                loss = self.loss_function(outputs, batch_y)
                loss.backward()

                # Gradient clipping
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)

                self.optimizer.step()
                train_losses.append(loss.item())

            # Validation
            self.model.eval()
            with torch.no_grad():
                val_outputs = self.model(self.X_val.to(self.device))
                val_loss = self.loss_function(val_outputs, self.y_val_norm.to(self.device))

                # Convert back to original scale for meaningful metrics
                val_outputs_orig = torch.exp(val_outputs * self.y_std + self.y_mean) - 1
                mae = torch.abs(val_outputs_orig - self.y_val.to(self.device)).mean()

            avg_train_loss = np.mean(train_losses)
            print(f"Epoch [{epoch}/{epochs}]")
            print(f"Train Loss: {avg_train_loss:.4f}, Val Loss: {val_loss.item():.4f}")
            print(f"Val mean absolute error: ${mae.item():.2f}")
            print(f"Learning rate: {self.scheduler.get_last_lr()[0]:.6f}")

            self.scheduler.step()

    # Save .pth model to use it later
    def save(self, path):
        torch.save(self.model.state_dict(), path)

    # Load .pth model
    def load(self, path, device="mps"):
        self.model.load_state_dict(torch.load(path, map_location=device))
        self.model.to(self.device)

    # Predicting
    def inference(self, item):
        self.model.eval()
        with torch.no_grad():
            vector = self.vectorizer.transform([item.summary])
            vector = torch.FloatTensor(vector.toarray()).to(self.device)
            pred = self.model(vector)[0]
            result = torch.exp(pred * self.y_std + self.y_mean) - 1
            result = result.item()
        return max(0, result)

In [6]:
# Feature selection ---> solit dataset ---> encoding
runner = DeepNeuralNetworkRunner(train, val)
runner.setup()

Deep Neural Network created with 289,128,449 parameters
Using cpu


In [18]:
# Modeling
runner.train(epochs=5)

100%|██████████| 263/263 [21:52<00:00,  4.99s/it]


Epoch [1/5]
Train Loss: 3.1116, Val Loss: 0.6752
Val mean absolute error: $33.14
Learning rate: 0.001000


100%|██████████| 263/263 [18:08<00:00,  4.14s/it]


Epoch [2/5]
Train Loss: 0.7121, Val Loss: 1.9994
Val mean absolute error: $42.33
Learning rate: 0.000976


100%|██████████| 263/263 [37:59<00:00,  8.67s/it]


Epoch [3/5]
Train Loss: 0.6408, Val Loss: 0.5051
Val mean absolute error: $25.55
Learning rate: 0.000905


100%|██████████| 263/263 [48:36<00:00, 11.09s/it] 


Epoch [4/5]
Train Loss: 0.4848, Val Loss: 0.5867
Val mean absolute error: $27.25
Learning rate: 0.000794


100%|██████████| 263/263 [44:00<00:00, 10.04s/it]


Epoch [5/5]
Train Loss: 0.4257, Val Loss: 0.5005
Val mean absolute error: $26.38
Learning rate: 0.000655


In [19]:
# Save .pth model to use it later
runner.save('deep_neural_network.pth')

In [7]:
runner.load("deep_neural_network.pth", device=runner.device)

In [8]:
# Test dataset
def deep_neural_network(item):
    return runner.inference(item)

In [9]:
# Test single datapoint
deep_neural_network(items[0])

10.215961456298828

In [10]:
# Evaluation
evaluate(deep_neural_network, test)

  0%|          | 0/2000 [00:00<?, ?it/s]

$15 $1 $10 $4 $15 $13 $4 $3 $6 $6 $1 $3 $23 $31 $14 $5 $6 $4 $250 $258 $17 $5 $9 $19 $7 $8 $4 $65 $6 $7 $7 $15 $69 $555 $3 $56 $3 $6 $5 $2 $26 $196 $1 $19 $31 $8 $8 $6 $1 $0 $21 $11 $11 $5 $7 $18 $27 $22 $6 $17 $62 $8 $23 $0 $73 $105 $9 $5 $13 $2 $6 $0 $2 $102 $5 $2 $15 $13 $38 $212 $30 $26 $17 $248 $37 $0 $5 $10 $6 $5 $10 $0 $5 $11 $3 $6 $0 $9 $0 $1 $6 $38 $36 $10 $8 $11 $7 $14 $14 $14 $42 $10 $33 $178 $25 $32 $6 $67 $6 $22 $28 $4 $10 $8 $234 $26 $25 $6 $16 $29 $5 $2 $253 $87 $8 $0 $1 $2 $10 $1 $0 $0 $10 $3 $9 $4 $5 $15 $7 $25 $36 $26 $20 $4 $83 $14 $4 $6 $51 $5 $23 $24 $1 $6 $64 $248 $0 $1 $22 $3 $28 $27 $84 $160 $26 $64 $9 $3 $5 $3 $30 $9 $1 $12 $207 $10 $17 $14 $3 $8 $15 $2 $41 $8 $7 $13 $34 $6 $28 $14 $12 $16 $14 $83 $5 $18 $12 $57 $20 $13 $19 $20 $29 $3 $1 $120 $5 $7 $29 $27 $9 $6 $16 $37 $2 $23 $10 $1 $4 $9 $45 $11 $2 $4 $0 $15 $0 $5 $2 $99 $20 $8 $6 $9 $21 $3 $14 $12 $309 $181 $38 $29 $30 $2 $41 $455 $63 $7 $5 $29 $7 $6 $16 $0 $14 $3 $5 $2 $17 $74 $10 $43 $7 $4 $10 $10 $10 $7 $